<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/saisathwik2703/flyrank-ml-internship-starter.git
%cd /content/flyrank-ml-internship-starter
!ls data/raw

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (394/394), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 394 (delta 235), reused 310 (delta 176), pack-reused 0 (from 0)
Receiving objects: 100% (394/394), 1.97 MiB | 10.61 MiB/s, done.
Resolving deltas: 100% (235/235), done.
/content/flyrank-ml-internship-starter
content_refresh_anonymized.csv


In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Raw rows: {len(df):,}")
print(f"Raw columns: {len(df.columns)}")

Raw rows: 30,000
Raw columns: 44


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature-vector construction

I use the anonymized content-performance dataset and build the feature vector before modeling.

The target is `is_declining_label`, where a row is labeled 1 when
`trend_direction == "down"`.

I use numeric content, search, engagement, age, and position signals together
with categorical content/context features.

For heavy-tailed traffic variables, I create `log1p` versions of the 90-day
impression, click, session, and AI-session totals.

Numeric missing values are filled with 0 and categorical missing values are
filled with `"unknown"`, matching the project preparation pipeline.

The model feature vector contains 18 numeric features and 8 categorical features.
IDs and label-derived fields are kept outside the feature matrix.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# ML-05 — Section 1: Build the feature vector
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Locate the cloned repository
# ------------------------------------------------------------

REPO = "/content/flyrank-ml-internship-starter"
DATA_PATH = os.path.join(
    REPO,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

# Check that the repository exists
if not os.path.exists(REPO):
    raise FileNotFoundError(
        f"Repository not found: {REPO}\n"
        "Run the git clone command first."
    )

# Check that the dataset exists
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check that the repository was cloned correctly."
    )

print("Repository found:")
print(REPO)

print("\nDataset found:")
print(DATA_PATH)

# ------------------------------------------------------------
# 2. Load the dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

# ------------------------------------------------------------
# 3. Define the target
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# ------------------------------------------------------------
# 4. Create engineered numeric features
# ------------------------------------------------------------

df["log_impressions_90d"] = np.log1p(
    pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0)
)

df["log_clicks_90d"] = np.log1p(
    pd.to_numeric(df["clicks_90d"], errors="coerce").fillna(0)
)

df["log_sessions_90d"] = np.log1p(
    pd.to_numeric(df["sessions_90d"], errors="coerce").fillna(0)
)

df["log_ai_sessions_90d"] = np.log1p(
    pd.to_numeric(df["ai_sessions_90d"], errors="coerce").fillna(0)
)

# ------------------------------------------------------------
# 5. Final model feature lists
#
# These are the features used by the starter pipeline.
# ------------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

MODEL_FEATURES = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)

# ------------------------------------------------------------
# 6. Verify that every feature exists
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "The following model features are missing:\n"
        + "\n".join(missing_features)
    )

# ------------------------------------------------------------
# 7. Handle missing values
# ------------------------------------------------------------

# Numeric features
for feature in MODEL_NUMERIC_FEATURES:
    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

# Categorical features
for feature in MODEL_CATEGORICAL_FEATURES:
    df[feature] = (
        df[feature]
        .fillna("unknown")
        .astype(str)
        .replace({
            "": "unknown",
            "nan": "unknown",
            "None": "unknown"
        })
    )

# ------------------------------------------------------------
# 8. Build X and y
# ------------------------------------------------------------

X = df[MODEL_FEATURES].copy()
y = df["is_declining_label"].copy()

# ------------------------------------------------------------
# 9. Final checks
# ------------------------------------------------------------

assert X.shape[1] == 26
assert len(X) == len(y)

assert "is_declining_label" not in X.columns
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FEATURE VECTOR BUILT SUCCESSFULLY")
print("=" * 60)

print(f"Rows:                 {len(X):,}")
print(f"Numeric features:     {len(MODEL_NUMERIC_FEATURES)}")
print(f"Categorical features: {len(MODEL_CATEGORICAL_FEATURES)}")
print(f"Total features:       {X.shape[1]}")

print(f"\nDeclining rows:       {y.sum():,}")
print(f"Declining rate:       {y.mean():.3f}")

print("\nFeature vector shape:")
print(X.shape)

print("\nNumeric features:")
for feature in MODEL_NUMERIC_FEATURES:
    print(" -", feature)

print("\nCategorical features:")
for feature in MODEL_CATEGORICAL_FEATURES:
    print(" -", feature)

print("\nFeature vector preview:")
display(X.head())

Repository found:
/content/flyrank-ml-internship-starter

Dataset found:
/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv

Dataset loaded successfully.
Shape: (30000, 44)

FEATURE VECTOR BUILT SUCCESSFULLY
Rows:                 30,000
Numeric features:     18
Categorical features: 8
Total features:       26

Declining rows:       16,262
Declining rate:       0.542

Feature vector shape:
(30000, 26)

Numeric features:
 - search_volume
 - competition
 - cpc
 - word_count
 - char_count
 - log_impressions_90d
 - log_clicks_90d
 - log_sessions_90d
 - log_ai_sessions_90d
 - days_with_impressions
 - days_with_sessions
 - content_age_days
 - days_since_last_update
 - ctr
 - avg_position
 - engagement_rate
 - scroll_rate
 - ai_traffic_pct

Categorical features:
 - competition_level
 - content_type
 - main_intent
 - age_tier
 - freshness_tier
 - word_count_tier
 - impression_tier
 - position_tier

Feature vector preview:


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,scroll_rate,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,4.55,0.0,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,10.00,0.0,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,28.57,0.0,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,3.45,0.0,LOW,keyword article,commercial,365+,0-30,unknown,good,page_1
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,24.29,0.0,LOW,keyword article,informational,181-365,0-30,2000-3500,good,page_3_5


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes

The features are reviewed based on their meaning, missing-value handling, data type, and availability at prediction time.

The unit of prediction is a content-refresh decision. Therefore, a feature is valid only when the information would be available before the refresh decision is made.

Numeric features are handled using numeric preprocessing, while categorical features are handled separately. Missing values are identified before modeling and are handled consistently during preprocessing.

To reduce data leakage, features representing future outcomes, future actions, or information that would only become available after the prediction point should not be used as model inputs.

The table generated below documents the available features and their missingness, type, and prediction-time availability.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 2: FEATURE NOTES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD THE ACTUAL DATASET AVAILABLE IN COLAB
# ------------------------------------------------------------

DATA_PATH = "./flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())


# ============================================================
# STEP 1: CHECK DATA TYPES AND MISSING VALUES
# ============================================================

print("\n" + "=" * 80)
print("DATA TYPE AND MISSING VALUE CHECK")
print("=" * 80)

feature_summary = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Count": df.isna().sum().values,
    "Missing %": (
        df.isna().mean().values * 100
    ).round(2),
    "Unique Values": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

display(feature_summary)


# ============================================================
# STEP 2: IDENTIFY CATEGORICAL AND NUMERIC FEATURES
# ============================================================

categorical_features = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_features = df.select_dtypes(
    include=np.number
).columns.tolist()

print("\n" + "=" * 80)
print("FEATURE TYPES")
print("=" * 80)

print("\nCategorical features:")
for col in categorical_features:
    print(" -", col)

print("\nNumeric features:")
for col in numeric_features:
    print(" -", col)


# ============================================================
# STEP 3: CREATE FEATURE NOTES
# ============================================================

feature_notes = []

for col in df.columns:

    # --------------------------------------------------------
    # Determine type
    # --------------------------------------------------------

    if col in categorical_features:
        feature_type = "Categorical"
        categorical = "Yes"
    else:
        feature_type = "Numeric"
        categorical = "No"


    # --------------------------------------------------------
    # Missing-value handling
    # --------------------------------------------------------

    missing_count = int(df[col].isna().sum())

    if missing_count == 0:
        missing_handling = "No missing values observed."
    elif feature_type == "Numeric":
        missing_handling = (
            "Missing numeric values should be handled during "
            "preprocessing using numeric imputation."
        )
    else:
        missing_handling = (
            "Missing categorical values should be handled during "
            "preprocessing using categorical imputation."
        )


    # --------------------------------------------------------
    # Prediction-time availability
    # --------------------------------------------------------

    available_when = (
        "Use only if the value is known before the content-refresh "
        "decision is made; exclude if it contains future information."
    )


    # --------------------------------------------------------
    # Meaning
    # --------------------------------------------------------

    meaning = (
        "Dataset feature used to describe the content-refresh "
        "item or its historical state."
    )


    # --------------------------------------------------------
    # Store information
    # --------------------------------------------------------

    feature_notes.append({
        "Feature": col,
        "Meaning": meaning,
        "Type": feature_type,
        "Categorical?": categorical,
        "Missing Count": missing_count,
        "Missing %": round(
            (missing_count / len(df)) * 100, 2
        ),
        "Missing-value handling": missing_handling,
        "Available before prediction?": available_when
    })


# ============================================================
# STEP 4: DISPLAY FINAL FEATURE NOTES TABLE
# ============================================================

feature_notes_df = pd.DataFrame(feature_notes)

print("\n" + "=" * 80)
print("SECTION 2 — FEATURE NOTES")
print("=" * 80)

display(feature_notes_df)


# ============================================================
# STEP 5: SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print("Total features:", len(df.columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print(
    "Features containing missing values:",
    int((df.isna().sum() > 0).sum())
)

print("\nFeature availability rule:")
print(
    "Only information available before the prediction/decision "
    "point should be used as a model feature."
)

Dataset loaded successfully!
Dataset shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

DATA TYPE AND MISSING VALUE CHECK


,Feature,Data Type,Missing Count,Missing %,Unique Values
0,content_id,object,0,0.00,30000
1,client_id,object,0,0.00,32
2,search_volume,float64,2468,8.23,41
3,competition,float64,2468,8.23,101
4,competition_level,object,2610,8.70,3
5,cpc,float64,2468,8.23,915
6,content_type,object,0,0.00,3
7,main_intent,object,2374,7.91,4
8,word_count,float64,7699,25.66,5476
9,char_count,float64,7699,25.66,14839



FEATURE TYPES

Categorical features:
 - content_id
 - client_id
 - competition_level
 - content_type
 - main_intent
 - provider_used
 - model_used
 - age_tier
 - freshness_tier
 - word_count_tier
 - char_count_tier
 - impression_tier
 - position_tier
 - trend_direction

Numeric features:
 - search_volume
 - competition
 - cpc
 - word_count
 - char_count
 - impressions_90d
 - clicks_90d
 - pageviews_90d
 - sessions_90d
 - users_90d
 - engaged_sessions_90d
 - ai_sessions_90d
 - scroll_events_90d
 - days_with_impressions
 - days_with_sessions
 - impressions_last_30d
 - clicks_last_30d
 - sessions_last_30d
 - impressions_prev_30d
 - clicks_prev_30d
 - sessions_prev_30d
 - content_age_days
 - age_tier_order
 - days_since_last_update
 - ctr
 - avg_position
 - engagement_rate
 - scroll_rate
 - ai_traffic_pct
 - trend_pct

SECTION 2 — FEATURE NOTES


,Feature,Meaning,Type,Categorical?,Missing Count,Missing %,Missing-value handling,Available before prediction?
0,content_id,Dataset feature used to describe the content-r...,Categorical,Yes,0,0.00,No missing values observed.,Use only if the value is known before the cont...
1,client_id,Dataset feature used to describe the content-r...,Categorical,Yes,0,0.00,No missing values observed.,Use only if the value is known before the cont...
2,search_volume,Dataset feature used to describe the content-r...,Numeric,No,2468,8.23,Missing numeric values should be handled durin...,Use only if the value is known before the cont...
3,competition,Dataset feature used to describe the content-r...,Numeric,No,2468,8.23,Missing numeric values should be handled durin...,Use only if the value is known before the cont...
4,competition_level,Dataset feature used to describe the content-r...,Categorical,Yes,2610,8.70,Missing categorical values should be handled d...,Use only if the value is known before the cont...
5,cpc,Dataset feature used to describe the content-r...,Numeric,No,2468,8.23,Missing numeric values should be handled durin...,Use only if the value is known before the cont...
6,content_type,Dataset feature used to describe the content-r...,Categorical,Yes,0,0.00,No missing values observed.,Use only if the value is known before the cont...
7,main_intent,Dataset feature used to describe the content-r...,Categorical,Yes,2374,7.91,Missing categorical values should be handled d...,Use only if the value is known before the cont...
8,word_count,Dataset feature used to describe the content-r...,Numeric,No,7699,25.66,Missing numeric values should be handled durin...,Use only if the value is known before the cont...
9,char_count,Dataset feature used to describe the content-r...,Numeric,No,7699,25.66,Missing numeric values should be handled durin...,Use only if the value is known before the cont...



SUMMARY
Total features: 44
Numeric features: 30
Categorical features: 14
Features containing missing values: 13

Feature availability rule:
Only information available before the prediction/decision point should be used as a model feature.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

Before training the model, I performed a leakage check on the available features. The purpose of this check is to identify variables that could give the model information that would not actually be available at the time of prediction.

I specifically checked for:

* **Label-derived columns** – columns that directly represent or are calculated from the target.
* **Future-window information** – columns that may summarize events occurring after the prediction point.
* **Product or outcome flags** – columns that may encode the result of a future decision or action.
* **Suspicious feature names** – names containing terms such as `label`, `target`, `outcome`, `future`, `next`, `after`, `post`, `converted`, or similar indicators.

The checks below are screening tests rather than proof that leakage is impossible. A feature is retained only when its meaning and timing are consistent with information available before the prediction decision.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 3: THE LEAKAGE HUNT
# ============================================================

import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# LOAD THE ACTUAL DATASET AVAILABLE IN COLAB
# ------------------------------------------------------------

DATA_PATH = "./flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nNumber of columns:", len(df.columns))


# ============================================================
# 1. SHOW ALL COLUMNS
# ============================================================

print("\n" + "=" * 80)
print("1. ALL DATASET COLUMNS")
print("=" * 80)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:3}. {col}")


# ============================================================
# 2. SEARCH FOR SUSPICIOUS / LABEL-DERIVED COLUMN NAMES
# ============================================================

print("\n" + "=" * 80)
print("2. SUSPICIOUS COLUMN-NAME CHECK")
print("=" * 80)

suspicious_terms = [
    "label",
    "target",
    "outcome",
    "future",
    "next",
    "after",
    "post",
    "converted",
    "conversion",
    "response",
    "result",
    "actual",
    "ground_truth",
    "groundtruth",
    "y_",
    "_y",
    "leak",
    "score"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    matched_terms = [
        term for term in suspicious_terms
        if term in col_lower
    ]

    if matched_terms:
        suspicious_columns.append({
            "Column": col,
            "Matched terms": ", ".join(matched_terms)
        })


if suspicious_columns:

    suspicious_df = pd.DataFrame(suspicious_columns)

    print("Potentially suspicious columns found:")
    display(suspicious_df)

else:

    print("No suspicious column names found using the screening terms.")


# ============================================================
# 3. CHECK POSSIBLE LABEL / TARGET COLUMNS
# ============================================================

print("\n" + "=" * 80)
print("3. POSSIBLE LABEL / TARGET COLUMNS")
print("=" * 80)

possible_target_terms = [
    "label",
    "target",
    "outcome",
    "ground_truth",
    "groundtruth",
    "future_outcome"
]

possible_targets = []

for col in df.columns:

    col_lower = col.lower()

    if any(term in col_lower for term in possible_target_terms):
        possible_targets.append(col)


if possible_targets:

    for col in possible_targets:
        print(f"Potential target column: {col}")
        print("Unique values:", df[col].nunique(dropna=True))

        if df[col].nunique(dropna=True) <= 20:
            print("Values:", df[col].dropna().unique())

        print()

else:

    print("No obvious target/label column detected by column name.")


# ============================================================
# 4. CHECK FUTURE-WINDOW / TIMING FEATURES
# ============================================================

print("\n" + "=" * 80)
print("4. FUTURE / TIMING COLUMN CHECK")
print("=" * 80)

future_terms = [
    "future",
    "next",
    "after",
    "post",
    "later",
    "subsequent",
    "7d",
    "14d",
    "30d",
    "60d",
    "90d",
    "future_",
    "_future",
    "next_",
    "_next"
]

future_columns = []

for col in df.columns:

    col_lower = col.lower()

    matched_terms = [
        term for term in future_terms
        if term in col_lower
    ]

    if matched_terms:

        future_columns.append({
            "Column": col,
            "Matched timing terms": ", ".join(matched_terms)
        })


if future_columns:

    future_df = pd.DataFrame(future_columns)

    print("Potential future-window columns:")
    display(future_df)

else:

    print("No obvious future-window columns detected.")


# ============================================================
# 5. CHECK PRODUCT / OUTCOME FLAGS
# ============================================================

print("\n" + "=" * 80)
print("5. PRODUCT / OUTCOME FLAG CHECK")
print("=" * 80)

flag_terms = [
    "flag",
    "converted",
    "conversion",
    "success",
    "failed",
    "failure",
    "completed",
    "approved",
    "rejected",
    "clicked",
    "click",
    "purchased",
    "purchase",
    "renewed",
    "renewal",
    "accepted"
]

flag_columns = []

for col in df.columns:

    col_lower = col.lower()

    matched_terms = [
        term for term in flag_terms
        if term in col_lower
    ]

    if matched_terms:

        flag_columns.append({
            "Column": col,
            "Matched terms": ", ".join(matched_terms)
        })


if flag_columns:

    flag_df = pd.DataFrame(flag_columns)

    print("Potential product/outcome flag columns:")
    display(flag_df)

else:

    print("No obvious product/outcome flag columns detected.")


# ============================================================
# 6. CHECK HIGH-CARDINALITY COLUMNS
# ============================================================
# High-cardinality columns can sometimes contain IDs or other
# fields that should not be directly used as predictive features.

print("\n" + "=" * 80)
print("6. HIGH-CARDINALITY / ID-LIKE COLUMN CHECK")
print("=" * 80)

high_cardinality = []

for col in df.columns:

    unique_count = df[col].nunique(dropna=True)
    unique_ratio = unique_count / len(df)

    if unique_ratio > 0.90:

        high_cardinality.append({
            "Column": col,
            "Unique values": unique_count,
            "Unique ratio": round(unique_ratio, 4)
        })


if high_cardinality:

    high_cardinality_df = pd.DataFrame(high_cardinality)

    print("Columns with more than 90% unique values:")
    display(high_cardinality_df)

else:

    print("No columns with more than 90% unique values.")


# ============================================================
# 7. CHECK COLUMNS THAT ARE CONSTANT
# ============================================================

print("\n" + "=" * 80)
print("7. CONSTANT COLUMN CHECK")
print("=" * 80)

constant_columns = []

for col in df.columns:

    if df[col].nunique(dropna=False) <= 1:
        constant_columns.append(col)


if constant_columns:

    print("Constant columns:")
    for col in constant_columns:
        print(" -", col)

else:

    print("No constant columns found.")


# ============================================================
# 8. AUTOMATIC LEAKAGE SCREENING SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("8. LEAKAGE SCREENING SUMMARY")
print("=" * 80)

print(
    "Suspicious-name columns:",
    len(suspicious_columns)
)

print(
    "Possible target/label columns:",
    len(possible_targets)
)

print(
    "Potential future/timing columns:",
    len(future_columns)
)

print(
    "Potential product/outcome flags:",
    len(flag_columns)
)

print(
    "High-cardinality columns:",
    len(high_cardinality)
)

print(
    "Constant columns:",
    len(constant_columns)
)


# ============================================================
# 9. FINAL PUBLIC-SAFE LEAKAGE STATEMENT
# ============================================================

print("\n" + "=" * 80)
print("FINAL LEAKAGE CHECK")
print("=" * 80)

print(
    "Leakage screening completed."
)

print(
    "Features that represent labels, future outcomes, future "
    "windows, or post-decision information should be excluded "
    "from model inputs."
)

print(
    "A feature is retained only when it is available before "
    "the prediction decision."
)

print(
    "Automated name-based checks are screening tests; final "
    "leakage decisions require checking the meaning and timing "
    "of each suspicious feature."
)

Dataset loaded successfully!
Dataset shape: (30000, 44)

Number of columns: 44

1. ALL DATASET COLUMNS
  1. content_id
  2. client_id
  3. search_volume
  4. competition
  5. competition_level
  6. cpc
  7. content_type
  8. main_intent
  9. word_count
 10. char_count
 11. provider_used
 12. model_used
 13. impressions_90d
 14. clicks_90d
 15. pageviews_90d
 16. sessions_90d
 17. users_90d
 18. engaged_sessions_90d
 19. ai_sessions_90d
 20. scroll_events_90d
 21. days_with_impressions
 22. days_with_sessions
 23. impressions_last_30d
 24. clicks_last_30d
 25. sessions_last_30d
 26. impressions_prev_30d
 27. clicks_prev_30d
 28. sessions_prev_30d
 29. content_age_days
 30. age_tier
 31. age_tier_order
 32. days_since_last_update
 33. freshness_tier
 34. word_count_tier
 35. char_count_tier
 36. ctr
 37. avg_position
 38. engagement_rate
 39. scroll_rate
 40. ai_traffic_pct
 41. impression_tier
 42. position_tier
 43. trend_direction
 44. trend_pct

2. SUSPICIOUS COLUMN-NAME CHECK
No sus

,Column,Matched timing terms
0,impressions_90d,90d
1,clicks_90d,90d
2,pageviews_90d,90d
3,sessions_90d,90d
4,users_90d,90d
5,engaged_sessions_90d,90d
6,ai_sessions_90d,90d
7,scroll_events_90d,90d
8,impressions_last_30d,30d
9,clicks_last_30d,30d



5. PRODUCT / OUTCOME FLAG CHECK
Potential product/outcome flag columns:


,Column,Matched terms
0,clicks_90d,click
1,clicks_last_30d,click
2,clicks_prev_30d,click



6. HIGH-CARDINALITY / ID-LIKE COLUMN CHECK
Columns with more than 90% unique values:


,Column,Unique values,Unique ratio
0,content_id,30000,1.0



7. CONSTANT COLUMN CHECK
No constant columns found.

8. LEAKAGE SCREENING SUMMARY
Suspicious-name columns: 0
Possible target/label columns: 0
Potential future/timing columns: 14
Potential product/outcome flags: 3
High-cardinality columns: 1
Constant columns: 0

FINAL LEAKAGE CHECK
Leakage screening completed.
Features that represent labels, future outcomes, future windows, or post-decision information should be excluded from model inputs.
A feature is retained only when it is available before the prediction decision.
Automated name-based checks are screening tests; final leakage decisions require checking the meaning and timing of each suspicious feature.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

I excluded fields that could introduce data leakage, encode future information, act as identifiers, or represent outcomes that would not be known at prediction time.

Each excluded field has a documented reason so that the final feature set remains reproducible and aligned with the prediction-time decision.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 4: WHAT I EXCLUDED AND WHY
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD THE DATASET
# ------------------------------------------------------------

DATA_PATH = "./flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)


# ============================================================
# STEP 1: DEFINE POSSIBLE EXCLUSION RULES
# ============================================================

# These are fields that may contain information unavailable
# at the prediction point or may introduce leakage.

leakage_terms = [
    "label",
    "target",
    "outcome",
    "future",
    "next",
    "after",
    "post",
    "ground_truth",
    "groundtruth",
    "converted",
    "conversion",
    "success",
    "failed",
    "failure",
    "completed",
    "approved",
    "rejected",
    "clicked",
    "click",
    "purchased",
    "purchase",
    "renewed",
    "renewal",
    "accepted"
]

id_terms = [
    "id",
    "uuid",
    "guid",
    "identifier"
]


# ============================================================
# STEP 2: FIND CANDIDATE EXCLUDED FIELDS
# ============================================================

excluded_fields = []

for col in df.columns:

    col_lower = col.lower()

    reason = None

    # --------------------------------------------------------
    # Check for label / future / outcome information
    # --------------------------------------------------------

    matched_leakage_terms = [
        term for term in leakage_terms
        if term in col_lower
    ]

    if matched_leakage_terms:

        reason = (
            "Excluded because the field may contain label-derived, "
            "future, or outcome information that may not be available "
            "at prediction time."
        )


    # --------------------------------------------------------
    # Check for identifier-like fields
    # --------------------------------------------------------

    if reason is None:

        matched_id_terms = [
            term for term in id_terms
            if term in col_lower
        ]

        if matched_id_terms:

            reason = (
                "Excluded because it appears to be an identifier "
                "rather than a meaningful predictive feature."
            )


    # --------------------------------------------------------
    # Add to exclusion list
    # --------------------------------------------------------

    if reason is not None:

        excluded_fields.append({
            "Excluded field": col,
            "Why excluded": reason
        })


# ============================================================
# STEP 3: REMOVE DUPLICATE EXCLUSIONS
# ============================================================

excluded_df = pd.DataFrame(excluded_fields)

if len(excluded_df) > 0:
    excluded_df = excluded_df.drop_duplicates(
        subset=["Excluded field"]
    ).reset_index(drop=True)


# ============================================================
# STEP 4: DISPLAY EXCLUDED FIELDS
# ============================================================

print("\n" + "=" * 80)
print("SECTION 4 — EXCLUDED FIELDS")
print("=" * 80)

if len(excluded_df) > 0:

    display(excluded_df)

else:

    print("No fields were automatically identified for exclusion.")


# ============================================================
# STEP 5: CREATE FINAL FEATURE LIST
# ============================================================

excluded_columns = excluded_df["Excluded field"].tolist() \
    if len(excluded_df) > 0 else []

final_features = [
    col for col in df.columns
    if col not in excluded_columns
]


# ============================================================
# STEP 6: SHOW BEFORE / AFTER
# ============================================================

print("\n" + "=" * 80)
print("FEATURE SET SUMMARY")
print("=" * 80)

print("Original number of columns:", len(df.columns))
print("Excluded columns:", len(excluded_columns))
print("Remaining columns:", len(final_features))


# ============================================================
# STEP 7: PRINT EXCLUDED FIELD + ONE-LINE REASON
# ============================================================

print("\n" + "=" * 80)
print("EXCLUSION DECISIONS")
print("=" * 80)

if len(excluded_df) > 0:

    for _, row in excluded_df.iterrows():

        print(
            f"❌ {row['Excluded field']}: "
            f"{row['Why excluded']}"
        )

else:

    print("No automatic exclusions found.")


# ============================================================
# STEP 8: SAVE THE EXCLUSION LIST
# ============================================================

excluded_df.to_csv(
    "excluded_features.csv",
    index=False
)

print("\nExclusion list saved as:")
print("excluded_features.csv")


# ============================================================
# STEP 9: FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("FINAL CHECK")
print("=" * 80)

print(
    "Excluded fields are not included in the proposed "
    "prediction feature set."
)

print(
    "Final feature count:",
    len(final_features)
)

Dataset loaded successfully!
Original shape: (30000, 44)

SECTION 4 — EXCLUDED FIELDS


,Excluded field,Why excluded
0,content_id,Excluded because it appears to be an identifie...
1,client_id,Excluded because it appears to be an identifie...
2,provider_used,Excluded because it appears to be an identifie...
3,clicks_90d,Excluded because the field may contain label-d...
4,clicks_last_30d,Excluded because the field may contain label-d...
5,clicks_prev_30d,Excluded because the field may contain label-d...



FEATURE SET SUMMARY
Original number of columns: 44
Excluded columns: 6
Remaining columns: 38

EXCLUSION DECISIONS
❌ content_id: Excluded because it appears to be an identifier rather than a meaningful predictive feature.
❌ client_id: Excluded because it appears to be an identifier rather than a meaningful predictive feature.
❌ provider_used: Excluded because it appears to be an identifier rather than a meaningful predictive feature.
❌ clicks_90d: Excluded because the field may contain label-derived, future, or outcome information that may not be available at prediction time.
❌ clicks_last_30d: Excluded because the field may contain label-derived, future, or outcome information that may not be available at prediction time.
❌ clicks_prev_30d: Excluded because the field may contain label-derived, future, or outcome information that may not be available at prediction time.

Exclusion list saved as:
excluded_features.csv

FINAL CHECK
Excluded fields are not included in the proposed predict

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.